In [40]:
import pandas as pd
import numpy as numpy

df = pd.read_csv("Missing_treated.csv")

In [41]:
df.shape

(57303, 52)

# Feature Selection

In [42]:
import pandas as pd
import numpy as np

target = 'construction_cost'

# Check target distribution
print("Construction cost summary:")
display(df[target].describe())

print("\nMissing target values:", df[target].isna().sum())
print("Zero or negative costs:", (df[target] <= 0).sum())

# Calculate correlations with numeric columns
numeric_df = df.select_dtypes(include='number')

correlations = (
    numeric_df.corr(method='spearman')[target]
    .drop(target)
    .dropna()
    .sort_values(key=abs, ascending=False)
)

print("\nNumeric features most associated with construction cost:")
display(correlations.to_frame('spearman_correlation').head(20))

Construction cost summary:


count    5.730300e+04
mean     1.577646e+05
std      6.753971e+05
min      0.000000e+00
25%      1.000000e+04
50%      1.500000e+04
75%      3.500000e+04
max      5.000000e+07
Name: construction_cost, dtype: float64


Missing target values: 0
Zero or negative costs: 173

Numeric features most associated with construction cost:


,spearman_correlation
machinery_cost,0.509724
scheme_depth,0.480832
construction_subsidy,0.349559
longitude,0.345406
culturable_command_area,0.328999
machinery_subsidy,0.315329
total_owners_holding,-0.284763
ipc_kharif_season,0.279857
ipc_total,0.274383
s_no_of_scheme,-0.229078


In [43]:
leakage_cols = [
    'machinery_cost',
    'construction_subsidy',
    'machinery_subsidy',
    'maintenance_cost'
]
# these are calculated after cost
df = df.drop(columns=leakage_cols, errors='ignore')

In [44]:
for col in [
    'scheme_status',
    'under_utilised_selection',
    'captured_in_previous_census'
]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        display(
            df.groupby(col, dropna=False)['construction_cost']
              .agg(
                  count='count',
                  median='median',
                  mean='mean'
              )
              .sort_values('count', ascending=False)
        )


--- scheme_status ---


,count,median,mean
scheme_status,,,
In-Use,51194,15000.0,134894.193200
Temporarily Not in use,5342,35000.0,325726.283710
Permanently Not in use,767,230000.0,514451.840626



--- under_utilised_selection ---


,count,median,mean
under_utilised_selection,,,
No,42718,15000.0,144530.243718
Yes,8546,12000.0,89322.120490
Unknown,6039,36000.0,348236.115774



--- captured_in_previous_census ---


,count,median,mean
captured_in_previous_census,,,
No,55322,15000.0,155485.802329
Yes,1144,20000.0,284758.821696
Unknown,837,10000.0,134812.482127


In [45]:
ipc_ipu_cols = [
    c for c in [
        'ipc_kharif_season',
        'ipc_rabi_season',
        'ipc_perennial_crop',
        'ipc_other_season',
        'ipc_total',
        'ipu_kharif_season',
        'ipu_rabi_season',
        'ipu_perennial_crop',
        'ipu_other_season',
        'ipu_total'
    ]
    if c in df.columns
]

display(
    df[ipc_ipu_cols].describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
)

,count,mean,std,min,25%,50%,75%,95%,99%,max
ipc_kharif_season,57303.0,2.155317,4.032863,0.0,0.666,1.5,2.1,4.0,20.0,250.0
ipc_rabi_season,57303.0,1.941574,2.717230,0.0,1.000,1.5,2.0,4.0,10.0,122.0
ipc_perennial_crop,57303.0,0.174485,2.885272,0.0,0.000,0.0,0.0,1.0,2.0,250.0
ipc_other_season,57303.0,0.375635,4.587932,0.0,0.000,0.0,0.2,2.0,3.5,750.0
ipc_total,57303.0,4.647010,8.718665,0.0,2.000,3.5,5.0,9.0,40.0,750.0
ipu_kharif_season,57303.0,1.801973,3.275841,0.0,0.500,1.0,2.0,4.0,10.0,250.0
ipu_rabi_season,57303.0,1.587813,1.934804,0.0,1.000,1.2,2.0,4.0,8.0,122.0
ipu_perennial_crop,57303.0,0.153559,2.850640,0.0,0.000,0.0,0.0,0.6,1.5,250.0
ipu_other_season,57303.0,0.288059,4.536090,0.0,0.000,0.0,0.1,1.5,2.9,750.0
ipu_total,57303.0,3.831123,7.500362,0.0,1.800,3.0,4.4,8.0,25.0,750.0


In [46]:
cols_to_check = [
    c for c in [
        'ipc_kharif_season', 'ipc_rabi_season',
        'ipc_perennial_crop', 'ipc_other_season', 'ipc_total',
        'ipu_kharif_season', 'ipu_rabi_season',
        'ipu_perennial_crop', 'ipu_other_season', 'ipu_total'
    ]
    if c in df.columns
]

display(df[cols_to_check].corr().round(2))

,ipc_kharif_season,ipc_rabi_season,ipc_perennial_crop,ipc_other_season,ipc_total,ipu_kharif_season,ipu_rabi_season,ipu_perennial_crop,ipu_other_season,ipu_total
ipc_kharif_season,1.00,0.79,0.05,0.05,0.75,0.82,0.60,0.04,0.02,0.54
ipc_rabi_season,0.79,1.00,0.04,0.07,0.73,0.61,0.74,0.02,0.04,0.49
ipc_perennial_crop,0.05,0.04,1.00,0.02,0.37,0.05,0.04,0.99,0.01,0.41
ipc_other_season,0.05,0.07,0.02,1.00,0.57,0.03,0.06,0.01,0.99,0.63
ipc_total,0.75,0.73,0.37,0.57,1.00,0.60,0.55,0.36,0.55,0.87
ipu_kharif_season,0.82,0.61,0.05,0.03,0.60,1.00,0.74,0.05,0.03,0.67
ipu_rabi_season,0.60,0.74,0.04,0.06,0.55,0.74,1.00,0.04,0.06,0.63
ipu_perennial_crop,0.04,0.02,0.99,0.01,0.36,0.05,0.04,1.00,0.01,0.42
ipu_other_season,0.02,0.04,0.01,0.99,0.55,0.03,0.06,0.01,1.00,0.64
ipu_total,0.54,0.49,0.41,0.63,0.87,0.67,0.63,0.42,0.64,1.00


In [47]:
extreme_cols = [
    'ipc_kharif_season',
    'ipc_rabi_season',
    'ipc_perennial_crop',
    'ipc_other_season',
    'ipc_total',
    'ipu_kharif_season',
    'ipu_rabi_season',
    'ipu_perennial_crop',
    'ipu_other_season',
    'ipu_total'
]

extreme_cols = [c for c in extreme_cols if c in df.columns]

display(
    df.loc[
        (df[extreme_cols] > 100).any(axis=1),
        extreme_cols + ['construction_cost']
    ].head(30)
)

,ipc_kharif_season,ipc_rabi_season,ipc_perennial_crop,ipc_other_season,ipc_total,ipu_kharif_season,ipu_rabi_season,ipu_perennial_crop,ipu_other_season,ipu_total,construction_cost
1404,150.000,0.0,0.000,0.0,150.00,0.000,0.0,0.000,0.0,0.00,90000.0
7979,0.000,0.0,250.000,0.0,250.00,0.000,0.0,250.000,0.0,250.00,800000.0
18820,0.000,0.0,138.000,0.0,138.00,0.000,0.0,138.000,0.0,138.00,2200000.0
19459,0.000,0.0,200.000,0.0,200.00,0.000,0.0,200.000,0.0,200.00,6000000.0
19462,0.000,0.0,200.000,0.0,200.00,0.000,0.0,200.000,0.0,200.00,6000000.0
19463,0.000,0.0,200.000,0.0,200.00,0.000,0.0,200.000,0.0,200.00,6000000.0
21523,100.000,50.0,0.000,0.0,150.00,100.000,50.0,0.000,0.0,150.00,50.0
21545,100.000,50.0,0.000,0.0,150.00,100.000,50.0,0.000,0.0,150.00,50.0
21556,100.000,50.0,0.000,0.0,150.00,100.000,50.0,0.000,0.0,150.00,50.0
23340,0.000,0.0,250.000,0.0,250.00,0.000,0.0,250.000,0.0,250.00,800000.0


In [48]:
# Check whether totals equal the sum of seasonal components
df['ipc_seasonal_sum'] = df[
    ['ipc_kharif_season', 'ipc_rabi_season',
     'ipc_perennial_crop', 'ipc_other_season']
].sum(axis=1)

df['ipu_seasonal_sum'] = df[
    ['ipu_kharif_season', 'ipu_rabi_season',
     'ipu_perennial_crop', 'ipu_other_season']
].sum(axis=1)

print(
    "IPC total mismatches:",
    (~np.isclose(df['ipc_total'], df['ipc_seasonal_sum'])).sum()
)

print(
    "IPU total mismatches:",
    (~np.isclose(df['ipu_total'], df['ipu_seasonal_sum'])).sum()
)

IPC total mismatches: 0
IPU total mismatches: 1


In [49]:
mismatch = ~np.isclose(df['ipu_total'], df['ipu_seasonal_sum'])

display(
    df.loc[mismatch, [
        'ipu_kharif_season',
        'ipu_rabi_season',
        'ipu_perennial_crop',
        'ipu_other_season',
        'ipu_total',
        'ipu_seasonal_sum',
        'construction_cost'
    ]]
)

,ipu_kharif_season,ipu_rabi_season,ipu_perennial_crop,ipu_other_season,ipu_total,ipu_seasonal_sum,construction_cost
45288,1.0,18.0,0.0,0.0,2.8,19.0,20000.0


In [50]:
cols_to_drop = [
    # Cost-related features that may leak the target
    'machinery_cost',
    'construction_subsidy',
    'machinery_subsidy',
    'maintenance_cost',

    # Post-construction operational status and outcomes
    'scheme_status',
    'under_utilised_selection',
    'captured_in_previous_census',

    # Operational history
    'pump_operating_days_kharif_season',
    'pump_operating_days_rabi_season',
    'pump_operating_days_perennial_crop',
    'pump_operating_days_other_season',
    'avg_pumping_per_day_kharif_season',
    'avg_pumping_per_day_rabi_season',
    'avg_pumping_per_day_perennial_crop',
    'avg_pumping_per_day_other_season',

    # Irrigation potential/utilisation measures
    'ipc_kharif_season',
    'ipc_rabi_season',
    'ipc_perennial_crop',
    'ipc_other_season',
    'ipc_total',
    'ipu_kharif_season',
    'ipu_rabi_season',
    'ipu_perennial_crop',
    'ipu_other_season',
    'ipu_total',

    # Temporary columns created during our checks
    'ipc_seasonal_sum',
    'ipu_seasonal_sum'
]

# Drop only columns that still exist
existing_cols = [col for col in cols_to_drop if col in df.columns]
df.drop(columns=existing_cols, inplace=True)

print("Columns dropped:", len(existing_cols))
print(existing_cols)
print("\nRemaining shape:", df.shape)

Columns dropped: 23
['scheme_status', 'under_utilised_selection', 'captured_in_previous_census', 'pump_operating_days_kharif_season', 'pump_operating_days_rabi_season', 'pump_operating_days_perennial_crop', 'pump_operating_days_other_season', 'avg_pumping_per_day_kharif_season', 'avg_pumping_per_day_rabi_season', 'avg_pumping_per_day_perennial_crop', 'avg_pumping_per_day_other_season', 'ipc_kharif_season', 'ipc_rabi_season', 'ipc_perennial_crop', 'ipc_other_season', 'ipc_total', 'ipu_kharif_season', 'ipu_rabi_season', 'ipu_perennial_crop', 'ipu_other_season', 'ipu_total', 'ipc_seasonal_sum', 'ipu_seasonal_sum']

Remaining shape: (57303, 27)


In [51]:
print("Dataset shape:", df.shape)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02}. {col} | dtype: {df[col].dtype} | "
          f"unique: {df[col].nunique(dropna=True)} | "
          f"missing: {df[col].isna().sum()}")

Dataset shape: (57303, 27)
01. district_name | dtype: object | unique: 34 | missing: 0
02. block_tehsil_name | dtype: object | unique: 227 | missing: 0
03. village_name | dtype: object | unique: 8393 | missing: 0
04. latitude | dtype: float64 | unique: 54532 | missing: 0
05. longitude | dtype: float64 | unique: 54650 | missing: 0
06. s_no_of_scheme | dtype: int64 | unique: 221 | missing: 0
07. scheme_type | dtype: object | unique: 2 | missing: 0
08. ownership | dtype: object | unique: 6 | missing: 0
09. total_owners_holding | dtype: float64 | unique: 305 | missing: 0
10. social_status | dtype: object | unique: 5 | missing: 0
11. gender | dtype: object | unique: 4 | missing: 0
12. commissioning_year | dtype: object | unique: 6 | missing: 0
13. scheme_depth | dtype: float64 | unique: 448 | missing: 0
14. scheme_diameter | dtype: float64 | unique: 295 | missing: 0
15. nearest_scheme_distance | dtype: float64 | unique: 277 | missing: 0
16. construction_cost | dtype: float64 | unique: 1271 

In [52]:
print("\ns_no_of_scheme — most frequent values:")
print(df['s_no_of_scheme'].value_counts().head(15))

print("\ns_no_of_scheme — sample values:")
print(sorted(df['s_no_of_scheme'].unique())[:30])

print("\nCommissioning year values:")
print(df['commissioning_year'].value_counts(dropna=False))

print("\nOperational data missing:")
print(df['operational_data_missing'].value_counts(dropna=False))


s_no_of_scheme — most frequent values:
s_no_of_scheme
1     8817
2     6046
3     4671
4     3821
5     3241
6     2808
7     2491
8     2205
9     1965
10    1762
11    1588
12    1414
13    1257
14    1137
15    1039
Name: count, dtype: int64

s_no_of_scheme — sample values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30)]

Commissioning year values:
commissioning_year
Up to 2018-2019     22215
During 2019-2020    11750
During 2020-2021     9626
During 2021-2022     7129
During 2022-2023     3534
During 2023-2024     3049
Name: count, dtype: int64

Operational data missing:
operational_data_missing
False    56163
True      114

In [53]:
cols_to_drop = [
    's_no_of_scheme',
    'commissioning_year',
    'operational_data_missing'
]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print("New shape:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

New shape: (57303, 24)

Remaining columns:
['district_name', 'block_tehsil_name', 'village_name', 'latitude', 'longitude', 'scheme_type', 'ownership', 'total_owners_holding', 'social_status', 'gender', 'scheme_depth', 'scheme_diameter', 'nearest_scheme_distance', 'construction_cost', 'water_distribution_method', 'lifting_device_type', 'energy_source', 'horse_power_of_lifting_device', 'located_in_command_area', 'only_recharge_ground_water', 'culturable_command_area', 'drinking_water_purpose', 'well_type', 'finance_source']


In [54]:
import numpy as np
import pandas as pd

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_features = [c for c in numeric_cols if c != 'construction_cost']

corr = df[numeric_features].corr(method='spearman').abs()

# Show only pairs with absolute Spearman correlation >= 0.85
high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)

high_corr_pairs.columns = ['Feature 1', 'Feature 2', 'Absolute Spearman Correlation']
high_corr_pairs = high_corr_pairs[
    high_corr_pairs['Absolute Spearman Correlation'] >= 0.85
].sort_values('Absolute Spearman Correlation', ascending=False)

display(high_corr_pairs)

,Feature 1,Feature 2,Absolute Spearman Correlation


In [55]:
categorical_cols = df.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

categorical_cols = [
    c for c in categorical_cols if c != 'construction_cost'
]

for col in categorical_cols:
    print(f"\n{'=' * 60}")
    print(f"{col} | Unique categories: {df[col].nunique(dropna=False)}")
    print(df[col].value_counts(dropna=False).head(15))

    summary = (
        df.groupby(col, dropna=False)['construction_cost']
          .agg(['count', 'median', 'mean'])
          .sort_values('count', ascending=False)
    )
    print("\nConstruction cost by category:")
    display(summary.head(15))


district_name | Unique categories: 34
district_name
Dhubri        14841
Barpeta       10315
Marigaon       5020
Bongaigaon     2922
Kamrup         2285
Goalpara       2273
Nagaon         2160
Darrang        2104
Golaghat       1272
Lakhimpur      1201
Kokrajhar      1149
Cachar         1067
Sonitpur       1064
Tinsukia        962
Hojai           927
Name: count, dtype: int64

Construction cost by category:


,count,median,mean
district_name,,,
Dhubri,14841,10000.0,26286.071364
Barpeta,10315,12000.0,29199.261137
Marigaon,5020,20000.0,103924.314044
Bongaigaon,2922,12000.0,49094.397077
Kamrup,2285,22000.0,506831.832328
Goalpara,2273,15000.0,74944.899147
Nagaon,2160,30000.0,181066.508565
Darrang,2104,25000.0,124844.141920
Golaghat,1272,27500.0,236269.866533



block_tehsil_name | Unique categories: 227
block_tehsil_name
Mandia              3429
Debitola            2475
Gauripur            2291
Mahamaya            2114
Pakabetbari         1791
Laharighat          1780
Rupshi              1473
Ruposhi             1292
Agomani             1273
Chenga              1194
Mayang              1159
Birshing-Jarua      1118
Dalgaon-Sialmari    1098
Barpeta              988
Nayeralga            920
Name: count, dtype: int64

Construction cost by category:


,count,median,mean
block_tehsil_name,,,
Mandia,3429,12000.0,19519.156244
Debitola,2475,17000.0,22173.986020
Gauripur,2291,25000.0,44342.842907
Mahamaya,2114,10000.0,17047.488108
Pakabetbari,1791,9000.0,31489.457845
Laharighat,1780,20000.0,63088.883146
Rupshi,1473,10000.0,22365.290815
Ruposhi,1292,12000.0,17217.752384
Agomani,1273,9000.0,18133.159764



village_name | Unique categories: 8393
village_name
Geramari Pt.I         221
Satra Kanara N.C.     200
Beguntali             165
Geramari Pt.Vi        120
Dalaner Alga Pt.Ii    112
Asharikandi           111
Majar Char N.C.       105
Sajuarkuti            105
Kawanbari             105
Aminerchor            103
Kachumara N.C.         97
Jhalar Alga Pt.I       92
Bowalguri              88
Alomganj Pt.Iv         87
Bouserkuti             86
Name: count, dtype: int64

Construction cost by category:


...

,count,median,mean
drinking_water_purpose,,,
No,55498,15000.0,148516.455982
Yes,1805,40000.0,442117.131247



well_type | Unique categories: 8
well_type
Tubewell: Shallow Tube well    47322
Tubewell: Medium Tube well      7761
Tubewell: Deep Tube well        1389
Dugwell: Dug well Pucca          477
Dugwell: Dug well Kutcha         228
Dugwell: Dug-cum-bore well        75
Dugwell: Others                   29
Tubewell: Artesian Well           22
Name: count, dtype: int64

Construction cost by category:


,count,median,mean
well_type,,,
Tubewell: Shallow Tube well,47322,13000.0,4.367440e+04
Tubewell: Medium Tube well,7761,437010.0,5.752585e+05
Tubewell: Deep Tube well,1389,500000.0,1.753466e+06
Dugwell: Dug well Pucca,477,50000.0,6.580397e+04
Dugwell: Dug well Kutcha,228,15000.0,2.556842e+04
Dugwell: Dug-cum-bore well,75,40000.0,2.558329e+05
Dugwell: Others,29,45000.0,5.546381e+05
Tubewell: Artesian Well,22,12000.0,4.508968e+04



finance_source | Unique categories: 18
finance_source
Others, Own savings              28579
Government fund                  10335
Own savings                       7793
Others                            3233
Owned by Group of farmers         3221
Government fund, Own savings      1433
Government fund, Others           1002
Money lender, Own savings          823
Money lender, Others               316
Bank loan, Own savings             244
Co-Operative Owned                 136
Panchayat Owned                     85
Bank loan, Others                   38
Bank loan, Government fund          26
Government fund, Money lender       16
Name: count, dtype: int64

Construction cost by category:


,count,median,mean
finance_source,,,
"Others, Own savings",28579,12000.0,17808.944863
Government fund,10335,427200.0,681765.583946
Own savings,7793,12000.0,17842.049365
Others,3233,15000.0,76439.561915
Owned by Group of farmers,3221,25000.0,169485.672201
"Government fund, Own savings",1433,72000.0,117104.451982
"Government fund, Others",1002,200000.0,294539.813772
"Money lender, Own savings",823,20000.0,35174.617254
"Money lender, Others",316,12000.0,13671.024051


In [56]:
# 1. Finance source vs ownership
print("Finance source vs ownership:")
display(pd.crosstab(
    df['ownership'],
    df['finance_source'],
    normalize='index'
).round(2))

# 2. Social status vs ownership
print("\nSocial status vs ownership:")
display(pd.crosstab(
    df['ownership'],
    df['social_status'],
    normalize='index'
).round(2))

# 3. Gender vs ownership
print("\nGender vs ownership:")
display(pd.crosstab(
    df['ownership'],
    df['gender'],
    normalize='index'
).round(2))

# 4. Rare categories in the remaining categorical features
categorical_cols = df.select_dtypes(
    include=['object', 'category', 'bool']
).columns.drop('construction_cost', errors='ignore')

for col in categorical_cols:
    counts = df[col].value_counts()
    print(
        f"{col}: {len(counts)} categories; "
        f"{(counts < 50).sum()} categories with fewer than 50 records"
    )

Finance source vs ownership:


finance_source,Bank loan,"Bank loan, Government fund","Bank loan, Money lender","Bank loan, Others","Bank loan, Own savings",Co-Operative Owned,Government fund,"Government fund, Money lender","Government fund, Others","Government fund, Own savings",Money lender,"Money lender, Others","Money lender, Own savings",Others,"Others, Own savings",Own savings,Owned by Group of farmers,Panchayat Owned
ownership,,,,,,,,,,,,,,,,,,
Co-Operative Owned,0.0,0.0,0.0,0.0,0.00,0.99,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.01,0.00,0.00,0.0
Govt. Owned,0.0,0.0,0.0,0.0,0.00,0.00,0.99,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.0
Others,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.99,0.00,0.00,0.00,0.0
Owned by Group of farmers,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.99,0.0
Owned by individual farmer,0.0,0.0,0.0,0.0,0.01,0.00,0.01,0.0,0.02,0.03,0.0,0.01,0.02,0.02,0.69,0.19,0.00,0.0
Panchayat Owned,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,1.0



Social status vs ownership:


social_status,OBC,Others,Scheduled-caste,Scheduled-tribe,Unknown
ownership,,,,,
Co-Operative Owned,0.01,0.01,0.00,0.00,0.99
Govt. Owned,0.00,0.00,0.00,0.00,0.99
Others,0.00,0.00,0.00,0.00,0.99
Owned by Group of farmers,0.00,0.00,0.00,0.00,0.99
Owned by individual farmer,0.11,0.77,0.05,0.07,0.00
Panchayat Owned,0.00,0.00,0.00,0.00,1.00



Gender vs ownership:


gender,Female,Male,Transgender,Unknown
ownership,,,,
Co-Operative Owned,0.00,0.01,0.0,0.99
Govt. Owned,0.00,0.01,0.0,0.99
Others,0.00,0.01,0.0,0.99
Owned by Group of farmers,0.00,0.01,0.0,0.99
Owned by individual farmer,0.01,0.99,0.0,0.00
Panchayat Owned,0.00,0.00,0.0,1.00


district_name: 34 categories; 0 categories with fewer than 50 records
block_tehsil_name: 227 categories; 62 categories with fewer than 50 records
village_name: 8393 categories; 8300 categories with fewer than 50 records
scheme_type: 2 categories; 0 categories with fewer than 50 records
ownership: 6 categories; 0 categories with fewer than 50 records
social_status: 5 categories; 0 categories with fewer than 50 records
gender: 4 categories; 1 categories with fewer than 50 records
water_distribution_method: 8 categories; 0 categories with fewer than 50 records
lifting_device_type: 6 categories; 0 categories with fewer than 50 records
energy_source: 8 categories; 2 categories with fewer than 50 records
located_in_command_area: 2 categories; 0 categories with fewer than 50 records
only_recharge_ground_water: 2 categories; 0 categories with fewer than 50 records
drinking_water_purpose: 2 categories; 0 categories with fewer than 50 records
well_type: 8 categories; 2 categories with fewer than

In [58]:
df.to_csv("feature_selected.csv", index=False)